# 02 - Extract JRC Production Route Data

## Purpose
Extract the CN code to production route mapping from JRC technical report
JRC134682, and hydrogen production route emission intensities from JRC135067.
These datasets provide production route granularity not available in the
CBAM default values xlsx.

## Inputs
- `data/raw/JRC134682_01.pdf` - Steel, aluminium, cement, fertilizers
- `data/raw/JRC135067_01.pdf` - Hydrogen (global average methodology)

## Outputs
- `data/processed/jrc_cn_production_routes.csv`
- `data/processed/jrc_hydrogen_routes.csv`

## Notes
- JRC134682 Table 1 (PDF pages 22-50, internal pages 18-46) maps each
  steel CN code to its possible production routes. Confirmed by manual
  inspection. Extraction stops at the footnote line beginning "*Numbers
  in the column Mapping..."
- JRC135067 Table 2 contains global average emission intensities per
  hydrogen production route. 6 rows, manually verified against source.
- Annex 2 of JRC134682 (country-level emission intensities) was assessed
  and determined to be redundant with the CBAM default values xlsx, which
  covers 119 countries vs the annex's 15-20, with the same direct/indirect/
  total columns. Not extracted.

## Section 1: JRC134682 Table 1 - CN Code to Production Route Mapping

Table 1 maps every CBAM-covered steel CN code to its possible production
routes: Primary route, Secondary EAF, Secondary Induction furnace, DRI
variants (Midrex, Corex, Rotary Kiln). This is the key join table between
CBAM default values and production route emission intensities.

### Structure Notes
- 4 columns: cn_code, mapping, production_route, description
- Section header rows: 21 rows with parent CN code and description but
  no production route. Dropped.
- Continuation rows: 13 rows where CN code groups overflowed across PDF
  rows. CN codes merged into previous valid row, continuation row dropped.
- Final clean dataset: 335 rows, 13 distinct production routes.

In [2]:
import pdfplumber
import pandas as pd
from pathlib import Path

pdf_path = Path("/Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/raw/JRC134682_01.pdf")

TABLE1_PAGES = range(21, 50)  # 0-indexed, PDF pages 22-50

all_rows = []
table_ended = False

with pdfplumber.open(pdf_path) as pdf:
    for pg_idx in TABLE1_PAGES:
        if table_ended:
            break
        page = pdf.pages[pg_idx]
        tables = page.extract_tables()
        for table in tables:
            if table_ended:
                break
            for row in table:
                # Stop at footnote line
                if any(cell and "*Numbers" in str(cell) for cell in row):
                    table_ended = True
                    break
                # Skip header rows
                if row[0] and "Product CN" in str(row[0]):
                    continue
                # Skip empty rows
                if not any(cell for cell in row):
                    continue
                # Only keep 4-column rows
                if len(row) != 4:
                    continue
                all_rows.append(row)

df = pd.DataFrame(all_rows, columns=["cn_code", "mapping", "production_route", "description"])

for col in df.columns:
    df[col] = df[col].str.replace("\n", " ", regex=False).str.strip()

df = df[df["cn_code"].notna() & (df["cn_code"] != "")]

print(f"Total rows extracted: {len(df)}")
print(f"\nSample:")
print(df.head(10).to_string())
print(f"\nUnique production routes:")
print(df["production_route"].unique())

Total rows extracted: 369

Sample:
           cn_code          mapping production_route                                                                                                                                                                                             description
0       2601 12 00       Sinter ore         Iron ore                                                                                                                                Agglomerated iron ores and concentrates, other than roasted iron pyrites
1       2601 12 00      Pellets ore         Iron ore                                                                                                                                Agglomerated iron ores and concentrates, other than roasted iron pyrites
2             7201             1, 9    Primary route                                                                                                                                        Pig iron and spieg

In [3]:
# Check empty production route rows
empty_routes = df[df["production_route"] == ""]
print(f"Rows with empty production route: {len(empty_routes)}")
print(empty_routes[["cn_code", "mapping", "production_route", "description"]].to_string())

Rows with empty production route: 34
                                                                                                           cn_code mapping production_route                                                                                                                                                description
10                                                                                                            7206                                                                                      Iron and non-alloy steel in ingots or other primary forms (excluding iron of heading no. 7203)
17                                                                                                            7207                                                                                                                             Iron or non-alloy steel; semi-finished products thereof
19                                                                            

In [4]:
# Identify and handle empty production route rows
# Type 1: section headers - have description but no production route, no mapping
# Type 2/3: continuation rows - CN code overflow from previous row

# Mark section header rows (no mapping, no production route, has description)
section_headers = (df["production_route"] == "") & (df["mapping"] == "") & (df["description"] != "")

# Mark continuation rows (no production route, no description)
continuation_rows = (df["production_route"] == "") & (df["description"] == "")

print(f"Section header rows: {section_headers.sum()}")
print(f"Continuation rows (no desc): {continuation_rows.sum()}")
print(f"Other empty production route rows: {(df['production_route'] == '').sum() - section_headers.sum() - continuation_rows.sum()}")

# Show any remaining cases
other_empty = df[(df["production_route"] == "") & ~section_headers & ~continuation_rows]
print(f"\nOther empty rows:")
print(other_empty.to_string())

Section header rows: 21
Continuation rows (no desc): 13
Other empty production route rows: 0

Other empty rows:
Empty DataFrame
Columns: [cn_code, mapping, production_route, description]
Index: []


In [5]:
# Step 1: Drop section header rows (no mapping, no production route, has description)
df_clean = df[~section_headers].copy()

# Step 2: For continuation rows, append their CN codes to the previous valid row
# then drop the continuation row itself
rows_to_drop = []

for idx in df_clean.index:
    if df_clean.loc[idx, "production_route"] == "" and df_clean.loc[idx, "description"] == "":
        # Find the previous valid row
        valid_rows = df_clean.loc[:idx-1]
        valid_rows = valid_rows[valid_rows["production_route"] != ""]
        if len(valid_rows) > 0:
            prev_idx = valid_rows.index[-1]
            # Append the overflow CN codes to the previous row
            df_clean.loc[prev_idx, "cn_code"] = (
                df_clean.loc[prev_idx, "cn_code"] + " " + df_clean.loc[idx, "cn_code"]
            ).strip()
        rows_to_drop.append(idx)

df_clean = df_clean.drop(rows_to_drop).reset_index(drop=True)

print(f"Rows after cleaning: {len(df_clean)}")
print(f"Empty production routes remaining: {(df_clean['production_route'] == '').sum()}")
print(f"\nUnique production routes:")
print(df_clean["production_route"].unique())
print(f"\nSample:")
print(df_clean.head(10).to_string())

Rows after cleaning: 335
Empty production routes remaining: 0

Unique production routes:
<StringArray>
[                                       'Iron ore',
                                   'Primary route',
                                    'Ferro-alloys',
                                    'DRI (Midrex)',
                                     'DRI (Corex)',
                           'Secondary route - EAF',
             'Secondary route - Induction furnace',
                                'DRI (Midrex)-EAF',
                                 'DRI (Corex)-BOF',
                           'DRI (Rotary Kiln)-EAF',
                       'Primary route steel alloy',
               'Secondary route steel alloy - EAF',
 'Secondary route steel alloy - Induction furnace']
Length: 13, dtype: str

Sample:
           cn_code          mapping production_route                                                                                                                                         

### Output
Saving cleaned Table 1 to `data/processed/jrc_cn_production_routes.csv`

In [6]:
output_path = Path("/Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/jrc_cn_production_routes.csv")
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Final shape: {df_clean.shape}")

Saved to: /Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/jrc_cn_production_routes.csv
Final shape: (335, 4)


## Section 2: JRC135067 Table 2 - Hydrogen Production Route Emission Intensities

Table 2 provides global average GHG emission intensities per hydrogen
production route. Unlike JRC134682 which provides country-level data,
this report estimates a single global average figure per route, reflecting
the current global production mix (based on 2021 IEA data).

### Structure Notes
- Source: JRC135067, internal page 6, PDF page 9
- 4 columns: feedstock_type, total_emissions_tco2_per_th2, comments, source
- 6 data rows covering: natural gas, coal, naphtha, oil,
  electrolysis chlor-alkali, electrolysis water
- No country breakdown. Global average only.
- Notable finding: water electrolysis at 23.1 tCO2/tH2 is higher than
  natural gas at 9.0, due to current global average grid emission intensity.
  This reverses as grids decarbonize.

In [1]:
import pandas as pd
from pathlib import Path
import pdfplumber

pdf_path = Path("/Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/raw/JRC135067_01.pdf")

# Table 2 is on internal page 6, PDF page 9 (0-indexed: page 8)
# "GHG emission intensities associated with the different hydrogen production routes"

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[8]
    tables = page.extract_tables()
    print(f"Tables found on page 9: {len(tables)}")
    for i, t in enumerate(tables):
        print(f"\nTable {i}: {len(t)} rows x {len(t[0])} cols")
        for row in t:
            print(f"  {row}")

Tables found on page 9: 1

Table 0: 6 rows x 2 cols
  ['9', '-']
  ['19.2', '-']
  ['7', 'This value considers the use of natural gas\ncovering the heat requirements of the process\nand substituting for exported hydrogen.']
  ['12', '-']
  ['7', 'This value considers the use of natural gas\ncovering the heat requirements of the process\nand substituting for exported hydrogen.']
  ['23.1', 'Despite the significant amount of total\nemissions associated to this pathway,\nelectrolysis has a negligible impact on the\nglobal average emission value because of its\nsmall global volumes.']


### Extraction Attempt
Text extraction was attempted using pdfplumber. The table structure in
JRC135067 page 9 causes pdfplumber to read across columns left to right,
mixing feedstock names, values and comments into a single text stream.
Table extraction returned only 2 of 4 columns with no header row.
Given the table has 6 rows with fully verifiable values, the DataFrame
is constructed directly from the extracted text output, with all values
cross-checked against the source document.

In [2]:
# Step 1: Extract raw text from page to document what pdfplumber returns
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[8]
    text = page.extract_text()

start_marker = "Table 2. GHG emission intensities associated with the different hydrogen production routes"
end_marker = "Source: As specified in the Table"

start_idx = text.find(start_marker)
end_idx = text.find(end_marker)

table_text = text[start_idx:end_idx].strip()
print("=== Raw extracted text ===")
print(table_text)

# Step 2: pdfplumber cannot reliably parse this table due to multi-column
# PDF rendering mixing feedstock names, values and comments in a single
# text stream. DataFrame constructed directly from verified source values.
# All figures cross-checked against JRC135067 internal page 6.

hydrogen_routes = pd.DataFrame([
    {"feedstock_type": "Natural gas",
     "total_emissions_tco2_per_th2": 9.0,
     "comments": "",
     "source": "IEA 2023"},
    {"feedstock_type": "Coal",
     "total_emissions_tco2_per_th2": 19.2,
     "comments": "",
     "source": "IEA 2023"},
    {"feedstock_type": "Naphtha (by-product)",
     "total_emissions_tco2_per_th2": 7.0,
     "comments": "Value considers natural gas covering heat requirements, substituting for exported hydrogen.",
     "source": "Lee & Elgowainy 2018"},
    {"feedstock_type": "Oil",
     "total_emissions_tco2_per_th2": 12.0,
     "comments": "",
     "source": "IEA 2019"},
    {"feedstock_type": "Electrolysis (chlor-alkali)",
     "total_emissions_tco2_per_th2": 7.0,
     "comments": "Value considers natural gas covering heat requirements, substituting for exported hydrogen.",
     "source": "Lee et al. 2018"},
    {"feedstock_type": "Electrolysis (water, world average)",
     "total_emissions_tco2_per_th2": 23.1,
     "comments": "High emissions due to current global grid mix. Negligible impact on global average given small production volumes.",
     "source": "IEA 2023"},
])

print("\n=== Constructed DataFrame ===")
print(hydrogen_routes.to_string())

=== Raw extracted text ===
Table 2. GHG emission intensities associated with the different hydrogen production routes
Total
emissions/ Comments
Feedstock
type tCO /tH Source
2 2
Natural Gas 9 - [3]
Coal 19.2 - [3]
This value considers the use of natural gas
7 covering the heat requirements of the process
Naphtha and substituting for exported hydrogen. [4]
Oil 12 - [5]
This value considers the use of natural gas
Electrolysis 7 covering the heat requirements of the process
(chlor-alkali) and substituting for exported hydrogen. [6]
Despite the significant amount of total
Electrolysis emissions associated to this pathway,
(water), 23.1 electrolysis has a negligible impact on the
world global average emission value because of its
average small global volumes. [3]

=== Constructed DataFrame ===
                        feedstock_type  total_emissions_tco2_per_th2                                                                                                            comments                

### CN Code for Hydrogen

Unlike steel and other CBAM materials which cover many distinct products
each with their own CN code, hydrogen has a single CN code under CBAM:
**2804 10 00**. 

Each CN code can be produced via multiple production routes,
but the code itself identifies the product, not the route.
Source: Annex I, Regulation (EU) 2023/956.
https://eur-lex.europa.eu/eli/reg/2023/956/oj/eng

CN code added as first column for consistency with other processed datasets
and to enable joins in the database layer.

In [3]:
# Add CN code as first column
hydrogen_routes.insert(0, "cn_code", "2804 10 00")
print(hydrogen_routes.to_string())

      cn_code                       feedstock_type  total_emissions_tco2_per_th2                                                                                                            comments                source
0  2804 10 00                          Natural gas                           9.0                                                                                                                                  IEA 2023
1  2804 10 00                                 Coal                          19.2                                                                                                                                  IEA 2023
2  2804 10 00                 Naphtha (by-product)                           7.0                         Value considers natural gas covering heat requirements, substituting for exported hydrogen.  Lee & Elgowainy 2018
3  2804 10 00                                  Oil                          12.0                                            

### Output
Saving cleaned Table 1 to `data/processed/jrc_cn_hydrogen_routes.csv`

In [4]:
output_path = Path("/Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/raw/JRC135067_01.pdf").parent.parent / "processed" / "jrc_hydrogen_routes.csv"
hydrogen_routes.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Final shape: {hydrogen_routes.shape}")

Saved to: /Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/jrc_hydrogen_routes.csv
Final shape: (6, 5)
